# 147 — Registro y promoción champion-challenger

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Δaccuracy = 0.928 − 0.914 = **+0.014 ≥ 0.01** ✓. Guarda p95: 198 ≤ 200
✓ (justo en el límite: vale la pena registrar que el margen es de 2 ms). Guarda nulas:
0.4 ≤ 0.5 ✓. **Se promueve**, dejando v_champion archivada y re-desplegable. Nota de
operación: un p95 a 2 ms del límite en sombra merece vigilancia post-promoción, porque
en tráfico real la latencia suele ser peor que en sombra.

**Ejercicio 2.** (1) **Ventanas distintas**: enero vs. octubre; la población y la
estacionalidad cambiaron, la diferencia puede ser del mundo y no del modelo.
(2) **Poblaciones/datasets distintos**: validación offline del challenger contra métrica
histórica de producción del champion; ni los datos ni el procedimiento de medición son
los mismos. La comparación válida es simultánea, sobre la misma población, con el mismo
pipeline de evaluación.

**Ejercicio 3.** `Registrada → Challenger → Archivada → Champion`. La transición
`Archivada → Champion` existe en el diagrama como **rollback de emergencia**, pero debe
exigir: que la versión conserve compatibilidad de firma con el pipeline actual, que sus
datos/features sigan disponibles, y una re-evaluación rápida post-restauración — un
modelo de hace seis meses puede estar obsoleto frente a la distribución actual (deriva,
clase 151).

**Ejercicio 4.** En el resultado del laboratorio, los pasos del flujo con sus salidas
estructuradas son la «evaluación» (hechos medidos) y el estado final del workflow es la
«decisión»; `limitations` recuerda que la decisión es didáctica, sin incertidumbre
muestral ni guardas de negocio reales.


In [ ]:
result = run_lab("workflow", seed=147)
assert result["kind"] == "workflow"
assert result["evidence"]
show(result)


In [ ]:
champion = {"accuracy": 0.914, "p95_ms": 145, "nulas_pct": 0.2}
challenger = {"accuracy": 0.928, "p95_ms": 198, "nulas_pct": 0.4}
DELTA_MIN = 0.01

delta = challenger["accuracy"] - champion["accuracy"]
checks = {
    "delta_min": delta >= DELTA_MIN,
    "guarda_p95": challenger["p95_ms"] <= 200,
    "guarda_nulas": challenger["nulas_pct"] <= 0.5,
}
print(f"delta = {delta:+.3f}")
for k, v in checks.items():
    print(f"{k}: {'OK' if v else 'VIOLADA'}")
print("PROMOVER" if all(checks.values()) else "MANTENER CHAMPION")


## Reflexión

1. ¿Por qué `delta_min > 0` en la regla de promoción, si cualquier mejora del AUC parece deseable? ¿Qué costos ignora un delta_min = 0?
2. En el ejemplo, v8 mejora el AUC global pero empeora un segmento: ¿qué mecanismo de la regla lo detectó y qué habría pasado evaluando solo la métrica primaria?
3. ¿Qué preguntas puede responder el shadow mode y cuáles solo puede responder un canario con tráfico real? Da un ejemplo de cada una.
